# SMART FHIR API Ingestion with Auto Loader

## Purpose

This notebook ingests synthetic healthcare data from the SMART Health IT
FHIR R4 API into the Bronze layer of the Khaoula Healthy Insurance Data Platform.

The ingestion process is intentionally separated into two responsibilities:

1. **API extraction** — retrieve paginated FHIR resources and land the raw
   payloads as newline-delimited JSON (NDJSON) files.
2. **Auto Loader ingestion** — incrementally discover newly landed files and
   append their raw resources to Bronze Delta tables.

### API Resources

- Patient
- Condition
- Encounter

### Landing Zone

`/Volumes/health_insurance/bronze/fhir_landing/`

### Bronze Targets

- `health_insurance.bronze.fhir_patient_raw`
- `health_insurance.bronze.fhir_condition_raw`
- `health_insurance.bronze.fhir_encounter_raw`

### Production Design

The Bronze layer preserves each FHIR resource as a raw JSON string. Auto Loader
reads the NDJSON landing files as text so that the original nested FHIR payload
is not flattened or re-shaped during ingestion.



##  Project and API Configuration

The project configuration is centralized so that catalog names, Volume paths,
resource limits, and target tables are not hard-coded throughout the notebook.

`PAGE_SIZE` controls the requested number of resources per API page.
`MAX_PAGES` limits the development extraction volume while still demonstrating
real pagination. Increasing these values later does not require changing the
ingestion architecture.


In [0]:

# Project and API configuration


CATALOG = "health_insurance"
BRONZE_SCHEMA = "bronze"

FHIR_BASE_URL = "https://r4.smarthealthit.org"

PAGE_SIZE = 100
MAX_PAGES = 5

FHIR_LANDING_BASE = (
    f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/fhir_landing"
)

AUTOLOADER_STATE_BASE = (
    f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/autoloader_state"
)

FHIR_TARGETS = {
    "Patient": {
        "landing_path": f"{FHIR_LANDING_BASE}/patient",
        "target_table": f"{CATALOG}.{BRONZE_SCHEMA}.fhir_patient_raw",
    },
    "Condition": {
        "landing_path": f"{FHIR_LANDING_BASE}/condition",
        "target_table": f"{CATALOG}.{BRONZE_SCHEMA}.fhir_condition_raw",
    },
    "Encounter": {
        "landing_path": f"{FHIR_LANDING_BASE}/encounter",
        "target_table": f"{CATALOG}.{BRONZE_SCHEMA}.fhir_encounter_raw",
    },
}

print("FHIR API:", FHIR_BASE_URL)
print("Page size:", PAGE_SIZE)
print("Maximum pages per resource:", MAX_PAGES)

for resource_type, config in FHIR_TARGETS.items():
    print(
        f"{resource_type:<10} | "
        f"{config['landing_path']} -> {config['target_table']}"
    )


## Validating the Required Unity Catalog Volumes

The Volumes are infrastructure and are created in the project setup notebook,
not here.

This ingestion notebook only verifies that the expected paths are available.
Keeping infrastructure creation in the setup stage makes the project easier to
reproduce and prevents ingestion code from silently changing the environment.


In [0]:

# Validating required Unity Catalog Volumes


required_volume_paths = [
    FHIR_LANDING_BASE,
    AUTOLOADER_STATE_BASE,
]

for volume_path in required_volume_paths:
    try:
        dbutils.fs.ls(volume_path)
        print(f"Available: {volume_path}")
    except Exception as exc:
        raise RuntimeError(
            f"Required Volume path is unavailable: {volume_path}. "
            "Run the project environment setup notebook first."
        ) from exc


##  Testing API Connectivity

Before starting an extraction batch, verify that Serverless compute can reach
the SMART Health IT FHIR endpoint and that the response is a FHIR search Bundle.

This is an ingestion health check only; it does not write any data.


In [0]:

# Test FHIR API connectivity


import requests

test_url = f"{FHIR_BASE_URL}/Patient?_count=3"

response = requests.get(
    test_url,
    headers={"Accept": "application/fhir+json"},
    timeout=30,
)

print("HTTP status:", response.status_code)

response.raise_for_status()

test_payload = response.json()

print("FHIR resource type:", test_payload.get("resourceType"))
print("Bundle type:", test_payload.get("type"))
print("Entries returned:", len(test_payload.get("entry", [])))


## Inspecting FHIR Pagination

FHIR search results are returned as Bundles. When additional pages are
available, the Bundle contains a link whose `relation` is `next`.

The ingestion logic follows that URL rather than attempting to construct
subsequent page URLs manually.


In [0]:

# Inspecting Bundle pagination


print("Top-level Bundle fields:")
print(list(test_payload.keys()))

print("\nBundle links:")

for link in test_payload.get("link", []):
    print(
        link.get("relation"),
        "->",
        link.get("url")
    )


def get_next_link(bundle):
    """Return the next-page URL from a FHIR Bundle, or None."""

    for link in bundle.get("link", []):
        if link.get("relation") == "next":
            return link.get("url")

    return None


print("\nNext page:")
print(get_next_link(test_payload))


##  Landing Raw FHIR API Pages as NDJSON

The API extraction stage writes raw resources to the landing Volume before
Bronze ingestion.

Each API page is persisted as an **NDJSON file** with one FHIR resource per
line. This has several benefits:

- the raw API payload is retained independently of Delta tables;
- extraction and Bronze ingestion are decoupled;
- Auto Loader can process newly arrived files incrementally;
- one API page is handled at a time instead of accumulating the entire
  extraction in driver memory;
- the content hash in the file name prevents an identical page payload from
  being written again during a simple rerun.

The hash protects against exact duplicate landing files. Record-level
deduplication/version handling still belongs in the Silver production pipeline,
where FHIR `id` and `meta.lastUpdated` can be used.


In [0]:

# Helpers for idempotent landing-file creation


import hashlib
import json


def landing_file_exists(path):
    """Return True when a landing file already exists."""

    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False


def write_fhir_page_to_landing(
    records,
    resource_type,
    landing_path,
):
    """
    Write one FHIR API page as NDJSON.

    A deterministic content hash is included in the file name so an
    identical page payload is not landed twice during a rerun.
    """

    if not records:
        return None

    json_lines = [
        json.dumps(
            record,
            sort_keys=True,
            separators=(",", ":"),
        )
        for record in records
    ]

    payload = "\n".join(json_lines)

    content_hash = hashlib.sha256(
        payload.encode("utf-8")
    ).hexdigest()[:20]

    filename = (
        f"{resource_type.lower()}_"
        f"{content_hash}.ndjson"
    )

    target_path = f"{landing_path}/{filename}"

    if landing_file_exists(target_path):
        print(
            f"Already landed, skipping: {target_path}"
        )
        return target_path

    dbutils.fs.put(
        target_path,
        payload,
        overwrite=False,
    )

    print(
        f"Landed {len(records):,} {resource_type} "
        f"resources -> {target_path}"
    )

    return target_path


##  Reusable Paginated API Extraction

The extraction function requests one page at a time, immediately writes that
page to the landing zone, and then follows the Bundle's `next` link.

This is more production-friendly than collecting all pages in a Python list
before writing anything because memory usage remains bounded by the page size.

`MAX_PAGES` is currently a deliberate portfolio/development limit. The same
function supports `max_pages=None` if a complete extraction is required later.


In [0]:

# Paginated FHIR extraction -> landing Volume


def extract_fhir_to_landing(
    resource_type,
    landing_path,
    page_size=100,
    max_pages=None,
):
    """
    Extract a FHIR resource type page-by-page and land each page as NDJSON.

    Returns summary information about the extraction.
    """

    dbutils.fs.mkdirs(landing_path)

    url = (
        f"{FHIR_BASE_URL}/{resource_type}"
        f"?_count={page_size}"
    )

    page_number = 0
    records_received = 0
    files_seen = []

    while url:
        page_number += 1

        print(
            f"Fetching {resource_type} "
            f"page {page_number}..."
        )

        response = requests.get(
            url,
            headers={"Accept": "application/fhir+json"},
            timeout=60,
        )

        response.raise_for_status()

        bundle = response.json()

        if bundle.get("resourceType") != "Bundle":
            raise ValueError(
                f"Expected a FHIR Bundle for {resource_type}, "
                f"received {bundle.get('resourceType')!r}."
            )

        records = [
            entry["resource"]
            for entry in bundle.get("entry", [])
            if entry.get("resource")
        ]

        records_received += len(records)

        landed_path = write_fhir_page_to_landing(
            records=records,
            resource_type=resource_type,
            landing_path=landing_path,
        )

        if landed_path:
            files_seen.append(landed_path)

        print(
            f"  Page records: {len(records):,} | "
            f"Total received: {records_received:,}"
        )

        if (
            max_pages is not None
            and page_number >= max_pages
        ):
            break

        url = get_next_link(bundle)

    return {
        "resource_type": resource_type,
        "pages_requested": page_number,
        "records_received": records_received,
        "landing_files": files_seen,
    }


## Extracting Patient, Condition, and Encounter Resources

This cell performs the API extraction stage for all configured resources.

Because landing files use deterministic content hashes, rerunning the same
unchanged extraction does not create an identical second landing file.


In [0]:

# Extracting configured FHIR resources to landing files


extraction_results = []

for resource_type, config in FHIR_TARGETS.items():

    print(f"\n{'=' * 70}")
    print(f"EXTRACTING: {resource_type}")
    print(f"{'=' * 70}")

    result = extract_fhir_to_landing(
        resource_type=resource_type,
        landing_path=config["landing_path"],
        page_size=PAGE_SIZE,
        max_pages=MAX_PAGES,
    )

    extraction_results.append(result)

print("\nExtraction summary:")

for result in extraction_results:
    print(
        f"{result['resource_type']:<10} | "
        f"pages={result['pages_requested']} | "
        f"records_received={result['records_received']:,}"
    )


## Inspecting the Landing Zone

This is descriptive ingestion profiling only. It verifies that raw files exist
for each resource before Auto Loader runs.

No business records are rejected or modified here. Formal data-quality rules
are attached later to the Bronze-to-Silver Lakeflow transformations.


In [0]:

# Inspect FHIR landing files


for resource_type, config in FHIR_TARGETS.items():

    files = dbutils.fs.ls(
        config["landing_path"]
    )

    data_files = [
        item
        for item in files
        if item.path.endswith(".ndjson")
    ]

    print(
        f"{resource_type:<10}: "
        f"{len(data_files)} landing file(s)"
    )


##  Auto Loader: Landing Files → Bronze Delta

Auto Loader owns the incremental file-to-Bronze step.

The landed NDJSON files are read with `cloudFiles.format = "text"`. Because
NDJSON stores one complete FHIR resource per line, the text reader returns one
resource in the `value` column. That value becomes `raw_json`.

This intentionally avoids parsing the FHIR structure in Bronze. The nested
payload remains intact for the existing Silver `from_json()` transformations.

A separate checkpoint is used for each resource. The checkpoint records which
files Auto Loader has already processed, so subsequent runs process only newly
discovered files.

`availableNow=True` processes all currently available new files and then stops;
the stream does not remain running continuously.


In [0]:
#
# Auto Loader: landing Volume -> Bronze Delta


from pyspark.sql import functions as F


def run_fhir_autoloader(
    resource_type,
    landing_path,
    target_table,
):
    """
    Incrementally ingest landed NDJSON files into a Bronze Delta table.

    FHIR JSON is preserved as a raw string. Only technical ingestion
    metadata is added.
    """

    resource_key = resource_type.lower()

    checkpoint_path = (
        f"{AUTOLOADER_STATE_BASE}/"
        f"{resource_key}/checkpoint"
    )

    source_df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "text")
        .load(landing_path)
    )

    bronze_df = (
        source_df
        .select(
            F.col("value").alias("raw_json"),
            F.current_timestamp().alias("_ingested_at"),
            F.lit("smart_fhir").alias("_source_system"),
            F.lit(resource_type).alias("_resource_type"),
            F.col("_metadata.file_name").alias("_source_file"),
            F.col("_metadata.file_modification_time").alias(
                "_source_file_modification_time"
            ),
        )
    )

    query = (
        bronze_df.writeStream
        .format("delta")
        .option(
            "checkpointLocation",
            checkpoint_path,
        )
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(target_table)
    )

    query.awaitTermination()

    print(
        f"Auto Loader completed: "
        f"{resource_type} -> {target_table}"
    )


## Running Auto Loader for All FHIR Resources

Run this after the one-time migration has been completed.

On later notebook executions, Auto Loader consults the per-resource checkpoint
and ignores files it has already processed.


In [0]:

# Running Auto Loader for all configured FHIR resources


for resource_type, config in FHIR_TARGETS.items():

    print(f"\n{'=' * 70}")
    print(f"AUTO LOADER: {resource_type}")
    print(f"{'=' * 70}")

    run_fhir_autoloader(
        resource_type=resource_type,
        landing_path=config["landing_path"],
        target_table=config["target_table"],
    )


## Verifying Bronze Ingestion

The following reconciliation confirms that all three Bronze tables exist and
contain data after Auto Loader finishes.

These counts are ingestion checks, not formal data-quality enforcement.


In [0]:
%sql

-- Verify Auto Loader Bronze tables


SELECT
    'Patient' AS resource,
    COUNT(*) AS bronze_records
FROM health_insurance.bronze.fhir_patient_raw

UNION ALL

SELECT
    'Condition',
    COUNT(*)
FROM health_insurance.bronze.fhir_condition_raw

UNION ALL

SELECT
    'Encounter',
    COUNT(*)
FROM health_insurance.bronze.fhir_encounter_raw;


In [0]:
%sql

-- Inspecting Bronze lineage metadata


SELECT
    _resource_type,
    _source_file,
    _source_file_modification_time,
    MIN(_ingested_at) AS first_ingested_at,
    MAX(_ingested_at) AS last_ingested_at,
    COUNT(*) AS records
FROM health_insurance.bronze.fhir_patient_raw
GROUP BY
    _resource_type,
    _source_file,
    _source_file_modification_time
ORDER BY _source_file;


## Ingestion Result

FHIR resources are now ingested through a production-oriented two-stage design.

### Source

SMART Health IT FHIR R4 API

Resources:

- Patient
- Condition
- Encounter

### Ingestion Flow

SMART Health IT FHIR API  
↓  
Paginated HTTP extraction  
↓  
Raw NDJSON files in Unity Catalog `fhir_landing` Volume  
↓  
Auto Loader (`cloudFiles`, text/NDJSON)  
↓  
Bronze Delta tables

### Bronze Tables

- `health_insurance.bronze.fhir_patient_raw`
- `health_insurance.bronze.fhir_condition_raw`
- `health_insurance.bronze.fhir_encounter_raw`

### Bronze Contract

Each Bronze row preserves one complete FHIR resource in:

- `raw_json`

Technical lineage fields are added:

- `_ingested_at`
- `_source_system`
- `_resource_type`
- `_source_file`
- `_source_file_modification_time`

No FHIR business fields are flattened or standardized in Bronze. The existing
Silver transformations continue to parse `raw_json` with `from_json()` and
produce structured Patient, Condition, and Encounter tables.

### Incremental Ingestion

Auto Loader maintains a separate checkpoint for each FHIR resource. Previously
processed landing files are therefore not reprocessed on subsequent runs.

The extraction helper also assigns deterministic content-hash file names so an
identical API page is not written to the landing zone twice during a rerun.

Record-level FHIR version/deduplication logic remains a Silver concern and can
use resource `id` and `meta.lastUpdated` when the transformations are wrapped
inside the production Lakeflow pipeline.

### Cost-Conscious Execution

`trigger(availableNow=True)` processes the currently available new files and
then terminates instead of leaving a continuous streaming query running.

### Data Quality Boundary

This notebook performs ingestion checks only.

Formal rules such as required resource identifiers, valid references,
chronological timestamps, allowed statuses, and warn/drop/fail behavior are
implemented as expectations while the Lakeflow Bronze-to-Silver transformations
produce validated Silver tables.



### Status

**FHIR API → Landing Volume → Auto Loader → Bronze ingestion: COMPLETE**
